In [170]:
import numpy as np
from numba import njit

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

import scipy.linalg as la

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.metrics import root_mean_squared_error, r2_score

import optuna

import warnings

# Init

In [3]:
steps = 20000

tau_steps = 1

transient_steps_henon = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_henon + transient_steps_reservoir + tau_steps
total_steps_after_henon = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

In [4]:
henon_dataset = np.zeros((total_steps, 2))

rng = np.random.default_rng(42)
henon_dataset[0] = rng.random(2)

a = 1.4
b = 0.3

In [5]:
@njit
def henon_numba(steps, a=1.4, b=0.3, x0=0.0, y0=0.0):
    X = np.zeros(steps)
    Y = np.zeros(steps)
    X[0] = x0
    Y[0] = y0

    for i in range(1, steps):
        X[i] = 1 - a * X[i - 1] ** 2 + Y[i - 1]
        Y[i] = b * X[i - 1]

    return X, Y

In [6]:
henon_data_x, henon_data_y = henon_numba(total_steps)

henon_dataset = np.column_stack((henon_data_x, henon_data_y))
henon_dataset = henon_dataset[transient_steps_henon:]

In [7]:
henon_scaler = StandardScaler()
henon_train_scaled = henon_scaler.fit_transform(henon_dataset[:-4000])
henon_test_scaled = henon_scaler.transform(henon_dataset[-4000:])
henon_scaled = np.concatenate((henon_train_scaled, henon_test_scaled), axis=0)

In [8]:
def henon_plot(data_list, names=None):
    fig = go.Figure()
    colors = ["white", "magenta"]

    for i, data in enumerate(data_list):
        fig.add_trace(
            go.Scattergl(
                x=data[:, 0],
                y=data[:, 1],
                mode="markers",
                name=names[i] if names else f"Dataset {i+1}",
                marker=dict(color=colors[i % len(colors)], size=1),
            )
        )

    fig.update_layout(
        plot_bgcolor="black",
        paper_bgcolor="black",
        font=dict(color="white"),
        xaxis=dict(
            showgrid=False,
            zeroline=False,
            linecolor="white",
            ticks="outside",
            tickcolor="white",
        ),
        yaxis=dict(
            showgrid=False,
            zeroline=False,
            linecolor="white",
            ticks="outside",
            tickcolor="white",
        ),
    )

    return fig

In [9]:
def r_2_plots_grid(actual_list, predicted_list, titles):
    fig = make_subplots(rows=1, cols=actual_list.shape[1], subplot_titles=titles)

    for i in range(actual_list.shape[1]):
        actual = actual_list[:, i]
        predicted = predicted_list[:, i]
        r_2 = r2_score(actual, predicted)
        col = i + 1

        fig.add_trace(go.Scatter(
            x=actual, y=predicted, mode="markers",
            name="Data", marker=dict(color="rgba(50, 50, 200, 0.5)", size=5)
        ), row=1, col=col)

        min_val, max_val = min(actual.min(), predicted.min()), max(actual.max(), predicted.max())
        fig.add_trace(go.Scatter(
            x=[min_val, max_val], y=[min_val, max_val], mode="lines", 
            name="Ideal", line=dict(color="firebrick", dash="dash")
        ), row=1, col=col)

        fig.update_xaxes(title_text="Actual", row=1, col=col)
        fig.update_yaxes(title_text=f"Predicted (R²: {r_2:.4f})", row=1, col=col)

    fig.update_layout(showlegend=False, height=500, width=1000)
    return fig

In [10]:
def generate_henon_grid(all_experiments, cols=3, plot_height=400):
    total_plots = len(all_experiments)
    rows = (total_plots + cols - 1) // cols
    colors = ["white", "magenta"]

    # 1. Initialize the master subplot matrix
    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=[f"System #{i+1}" for i in range(total_plots)],
        horizontal_spacing=0.04,
        vertical_spacing=0.03,
    )

    for idx, data_list in enumerate(all_experiments):
        current_row = (idx // cols) + 1
        current_col = (idx % cols) + 1

        for i, data in enumerate(data_list):
            fig.add_trace(
                go.Scattergl(
                    x=data[:, 0],
                    y=data[:, 1],
                    mode="markers",
                    marker=dict(color=colors[i % len(colors)], size=1),
                    showlegend=False,
                ),
                row=current_row,
                col=current_col,
            )

    fig.update_layout(
        height=plot_height * rows,
        plot_bgcolor="black",
        paper_bgcolor="black",
        font=dict(color="white", size=10),
        margin=dict(t=80, b=40, l=40, r=40),
    )

    fig.update_xaxes(
        showgrid=False,
        zeroline=False,
        linecolor="white",
        ticks="outside",
        tickcolor="white",
    )
    fig.update_yaxes(
        showgrid=False,
        zeroline=False,
        linecolor="white",
        ticks="outside",
        tickcolor="white",
    )

    return fig

# Bayesian Init

In [201]:
@njit(fastmath=True, cache=True)
def compute_states(steps, henon_inputs, res_size, W_in, W_res, bias, alpha, noise):
    X_data = np.zeros((steps, res_size))
    X_data[0] = 0.0

    state = np.zeros(res_size)
    input_projections = (henon_inputs + noise) @ W_in.T

    for i in range(1, steps):
        state = (1.0 - alpha) * state + alpha * np.tanh(
            input_projections[i - 1] + W_res @ state + bias
        )
        X_data[i] = state
    return X_data

In [12]:
@njit(fastmath=True, cache=True)
def compute_closed_states(
    steps_start,
    steps_end,
    Y_pred_scaled,
    X_pred,
    W_in,
    W_res,
    bias,
    alpha,
    W_out,
    W_bias,
    res_size,
    scaler_mean,
    scaler_std,
):
    for i in range(steps_start, steps_end):
        u = Y_pred_scaled[i - 2]
        prev_state = X_pred[i - 1, :res_size]
        new_state = np.tanh(W_in @ u + W_res @ prev_state + bias)
        X_pred[i, :res_size] = (1.0 - alpha) * prev_state + alpha * new_state
        X_scaled = (X_pred[i, :res_size] - scaler_mean) / scaler_std
        Y_pred_scaled[i] = W_out @ X_scaled + W_bias
    return Y_pred_scaled

In [13]:
@njit(fastmath=True, cache=True)
def lyapunov_loss(Y_test, Y_pred, a, b, total_steps_after_henon):
    Q_test = np.ascontiguousarray(np.eye(2))
    lyapunov_sums_test = np.zeros(2)
    for i in range(len(Y_test)):
        J = np.array([[-2.0 * a * Y_test[i, 0], 1.0], [b, 0.0]])
        Z = np.ascontiguousarray(J @ Q_test)
        Q_test_raw, R = np.linalg.qr(Z)
        Q_test = np.ascontiguousarray(Q_test_raw)
        lyapunov_sums_test += np.log(np.abs(np.diag(R)))
    lyapunov_exponent_test = lyapunov_sums_test / total_steps_after_henon

    Q_pred = np.ascontiguousarray(np.eye(2))
    lyapunov_sums_pred = np.zeros(2)
    for i in range(len(Y_pred)):
        J = np.array([[-2.0 * a * Y_pred[i, 0], 1.0], [b, 0.0]])
        Z = np.ascontiguousarray(J @ Q_pred)
        Q_pred_raw, R = np.linalg.qr(Z)
        Q_pred = np.ascontiguousarray(Q_pred_raw)
        lyapunov_sums_pred += np.log(np.abs(np.diag(R)))
    lyapunov_exponent_pred = lyapunov_sums_pred / total_steps_after_henon

    le_loss = (lyapunov_exponent_test - lyapunov_exponent_pred) ** 2
    return le_loss[0]

In [14]:
def normalize_2d(data):
    min_vals = data.min(axis=0)
    max_vals = data.max(axis=0)
    return (data - min_vals) / (max_vals - min_vals)

In [15]:
@njit(fastmath=True, cache=True)
def random_sparse_orthogonal(dim, density, rng):
    Q = np.eye(dim)
    target_nnz = int(dim**2 * density)
    check_frequency = max(1, dim // 40)
    for _ in range(dim**2):
        i = rng.integers(0, dim)
        j = rng.integers(0, dim)
        while i == j:
            j = rng.integers(0, dim)

        theta = rng.uniform(0, 2 * np.pi)
        c, s = np.cos(theta), np.sin(theta)

        for k in range(dim):
            temp_i = Q[i, k]
            temp_j = Q[j, k]
            Q[i, k] = c * temp_i + s * temp_j
            Q[j, k] = -s * temp_i + c * temp_j

        if _ % check_frequency == 0:
            if np.count_nonzero(Q) >= target_nnz:
                break
    return Q


N = 200
rng = np.random.default_rng(42)
sparse_orthogonal = random_sparse_orthogonal(dim=N, density=0.1, rng=rng)

print("Sample slice of random float values:")
print(sparse_orthogonal[:5, :5].round(2))

print(f"\nTotal non-zero elements: {np.count_nonzero(sparse_orthogonal)}")
print(f"Actual matrix density: {np.count_nonzero(sparse_orthogonal) / (N**2):.4f}")

product = sparse_orthogonal @ sparse_orthogonal.T
identity = np.eye(sparse_orthogonal.shape[0])
error_matrix = product - identity
orthogonality_error = np.linalg.norm(error_matrix, ord="fro")
print(f"Orthogonality Error: {orthogonality_error:.2e}")

Sample slice of random float values:
[[ 0.95  0.    0.    0.    0.  ]
 [-0.    0.01 -0.   -0.   -0.  ]
 [ 0.    0.   -0.11  0.    0.  ]
 [ 0.    0.    0.   -0.15  0.  ]
 [ 0.    0.    0.    0.    0.08]]

Total non-zero elements: 4041
Actual matrix density: 0.1010
Orthogonality Error: 2.77e-15


# Deep Reservoir Open

## Reservoirs

### Res 1

In [ ]:
res_1 = {
    "params": {
        "in_size": 2,
        "out_size": 2,
        "res_size": 200,
        "sparsity": 0.2,
        "alpha": 0.65,
        "bias_scaling": 0.1,
        "input_scaling": 0.4,
        "spec_rad": 1.0,
        "ridge_alpha": 1e-5
        "noise_val": 0,
    },
    "values": {},
    "data": {},
}

In [95]:
rng = np.random.default_rng(42)
res_1["values"]["bias"] = rng.uniform(
    -res_1["params"]["bias_scaling"],
    res_1["params"]["bias_scaling"],
    res_1["params"]["res_size"],
)
res_1["values"]["W_in"] = rng.uniform(
    -res_1["params"]["input_scaling"],
    res_1["params"]["input_scaling"],
    (res_1["params"]["res_size"], res_1["params"]["in_size"]),
)
W_res = rng.uniform(
    -1.0, 1.0, (res_1["params"]["res_size"], res_1["params"]["res_size"])
)
mask = (
    rng.random((res_1["params"]["res_size"], res_1["params"]["res_size"]))
    < res_1["params"]["sparsity"]
)
W_res *= mask
try:
    eigenvalues = la.eigvals(W_res)
except Exception as e:
    print("Eigenval Issue")
largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)
res_1["values"]["W_res"] = W_res * (res_1["params"]["spec_rad"] / largest_eigenvalue)
res_1["values"]["noise"] = rng.normal(
    0,
    res_1["params"]["noise_val"],
    size=(steps + transient_steps_reservoir, res_1["params"]["in_size"]),
)

In [96]:
res_1["data"]["state"] = compute_states(
    steps + transient_steps_reservoir + tau_steps,
    henon_scaled[:-tau_steps],
    res_1["params"]["res_size"],
    res_1["values"]["W_in"],
    res_1["values"]["W_res"],
    res_1["values"]["bias"],
    res_1["params"]["alpha"],
    res_1["values"]["noise"],
)

### Res 2

In [ ]:
res_2 = {
    "params": {
        "in_size": res_1["params"]["res_size"],
        "out_size": 2,
        "res_size": 300,
        "sparsity": 0.15,
        "alpha": 0.7,
        "bias_scaling": 0.2,
        "input_scaling": 0.2,
        "spec_rad": .95,
        "ridge_alpha": 1e-4,
        "noise_val": 0,
    },
    "values": {},
    "data": {},
}

In [98]:
rng = np.random.default_rng(42)
res_2["values"]["bias"] = rng.uniform(
    -res_2["params"]["bias_scaling"],
    res_2["params"]["bias_scaling"],
    res_2["params"]["res_size"],
)
res_2["values"]["W_in"] = rng.uniform(
    -res_2["params"]["input_scaling"],
    res_2["params"]["input_scaling"],
    (res_2["params"]["res_size"], res_2["params"]["in_size"]),
)
W_res = rng.uniform(
    -1.0, 1.0, (res_2["params"]["res_size"], res_2["params"]["res_size"])
)
mask = (
    rng.random((res_2["params"]["res_size"], res_2["params"]["res_size"]))
    < res_2["params"]["sparsity"]
)
W_res *= mask
try:
    eigenvalues = la.eigvals(W_res)
except Exception as e:
    print("Eigenval Issue")
largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)
res_2["values"]["W_res"] = W_res * (res_2["params"]["spec_rad"] / largest_eigenvalue)
res_2["values"]["noise"] = rng.normal(
    0,
    res_2["params"]["noise_val"],
    size=(steps + transient_steps_reservoir, res_2["params"]["in_size"]),
)

In [99]:
res_2["data"]["state"] = compute_states(
    steps + transient_steps_reservoir + tau_steps,
    res_1["data"]["state"][:-tau_steps],
    res_2["params"]["res_size"],
    res_2["values"]["W_in"],
    res_2["values"]["W_res"],
    res_2["values"]["bias"],
    res_2["params"]["alpha"],
    res_2["values"]["noise"],
)

### Res 3

In [140]:
res_3 = {
    "params": {
        "in_size": res_2["params"]["res_size"],
        "out_size": 2,
        "res_size": 200,
        "sparsity": 0.2,
        "alpha": 0.7,
        "bias_scaling": 0.07,
        "input_scaling": 0.2,
        "spec_rad": 1.05,
        "ridge_alpha": 1e-3,
        "noise_val": 0,
    },
    "values": {},
    "data": {},
}

In [141]:
rng = np.random.default_rng(42)
res_3["values"]["bias"] = rng.uniform(
    -res_3["params"]["bias_scaling"],
    res_3["params"]["bias_scaling"],
    res_3["params"]["res_size"],
)
res_3["values"]["W_in"] = rng.uniform(
    -res_3["params"]["input_scaling"],
    res_3["params"]["input_scaling"],
    (res_3["params"]["res_size"], res_3["params"]["in_size"]),
)
W_res = rng.uniform(
    -1.0, 1.0, (res_3["params"]["res_size"], res_3["params"]["res_size"])
)
mask = (
    rng.random((res_3["params"]["res_size"], res_3["params"]["res_size"]))
    < res_3["params"]["sparsity"]
)
W_res *= mask
try:
    eigenvalues = la.eigvals(W_res)
except Exception as e:
    print("Eigenval Issue")
largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)
res_3["values"]["W_res"] = W_res * (res_3["params"]["spec_rad"] / largest_eigenvalue)
res_3["values"]["noise"] = rng.normal(
    0,
    res_3["params"]["noise_val"],
    size=(steps + transient_steps_reservoir, res_3["params"]["in_size"]),
)

In [142]:
res_3["data"]["state"] = compute_states(
    steps + transient_steps_reservoir + tau_steps,
    res_2["data"]["state"][:-tau_steps],
    res_3["params"]["res_size"],
    res_3["values"]["W_in"],
    res_3["values"]["W_res"],
    res_3["values"]["bias"],
    res_3["params"]["alpha"],
    res_3["values"]["noise"],
)

## Regression

In [110]:
Y_data = henon_scaled[transient_steps_reservoir + tau_steps :]
Y_train_scaled, Y_test_scaled = (
    Y_data[:-test_steps],
    Y_data[-test_steps:],
)

### Res 1

In [ ]:
X_data = res_1["data"]["state"][transient_steps_reservoir:-tau_steps]

x_scaler = StandardScaler()
X_train_scaled, X_test_scaled = (
    x_scaler.fit_transform(X_data[:-test_steps]),
    x_scaler.transform(X_data[-test_steps:]),
)

In [101]:
try:
    model = Ridge(alpha=res_1["params"]["ridge_alpha"], solver="auto")
    model.fit(X_train_scaled, Y_train_scaled)
except Exception as e:
    try:
        model = Ridge(alpha=res_1["params"]["ridge_alpha"], solver="svd")
        model.fit(X_train_scaled, Y_train_scaled)
        print("Used SVD solver")
    except Exception as e:
        print(f"Error fitting model: {e}")

In [102]:
Y_pred_scaled = model.predict(X_test_scaled)

res_1["data"]["Y_pred"] = henon_scaler.inverse_transform(Y_pred_scaled)
Y_test = henon_scaler.inverse_transform(Y_test_scaled)

In [103]:
res_1["data"]["rmes"] = root_mean_squared_error(Y_test, res_1["data"]["Y_pred"])
res_1["data"]["r_2"] = r2_score(Y_test, res_1["data"]["Y_pred"])
res_1["data"]["rmes"], res_1["data"]["r_2"]

(0.0778276766447169, 0.9778026063389225)

In [104]:
henon_plot([Y_test, res_1["data"]["Y_pred"]], names=["Actual", "Predicted"])

### Res 2

In [ ]:
X_data = res_2["data"]["state"][transient_steps_reservoir:-tau_steps]

x_scaler = StandardScaler()
X_train_scaled, X_test_scaled = (
    x_scaler.fit_transform(X_data[:-test_steps]),
    x_scaler.transform(X_data[-test_steps:]),
)

In [123]:
try:
    model = Ridge(alpha=res_2["params"]["ridge_alpha"], solver="auto")
    model.fit(X_train_scaled, Y_train_scaled)
except Exception as e:
    try:
        model = Ridge(alpha=res_2["params"]["ridge_alpha"], solver="svd")
        model.fit(X_train_scaled, Y_train_scaled)
        print("Used SVD solver")
    except Exception as e:
        print(f"Error fitting model: {e}")

In [124]:
Y_pred_scaled = model.predict(X_test_scaled)

res_2["data"]["Y_pred"] = henon_scaler.inverse_transform(Y_pred_scaled)
Y_test = henon_scaler.inverse_transform(Y_test_scaled)

In [125]:
res_2["data"]["rmes"] = root_mean_squared_error(Y_test, res_2["data"]["Y_pred"])
res_2["data"]["r_2"] = r2_score(Y_test, res_2["data"]["Y_pred"])
res_2["data"]["rmes"], res_2["data"]["r_2"]

(0.28529389080102413, 0.7051813905557153)

In [126]:
henon_plot([Y_test, res_2["data"]["Y_pred"]], names=["Actual", "Predicted"])

### Res 1 and 2

In [135]:
X_data_1 = res_1["data"]["state"][transient_steps_reservoir:-tau_steps]
X_data_2 = res_2["data"]["state"][transient_steps_reservoir:-tau_steps]
X_data = np.hstack((X_data_1, X_data_2))

x_scaler = StandardScaler()
X_train_scaled, X_test_scaled = (
    x_scaler.fit_transform(X_data[:-test_steps]),
    x_scaler.transform(X_data[-test_steps:]),
)

In [136]:
try:
    model = Ridge(alpha=res_1["params"]["ridge_alpha"], solver="auto")
    model.fit(X_train_scaled, Y_train_scaled)
except Exception as e:
    try:
        model = Ridge(alpha=res_1["params"]["ridge_alpha"], solver="svd")
        model.fit(X_train_scaled, Y_train_scaled)
        print("Used SVD solver")
    except Exception as e:
        print(f"Error fitting model: {e}")

In [137]:
Y_pred_scaled = model.predict(X_test_scaled)

res_2["data"]["Y_pred"] = henon_scaler.inverse_transform(Y_pred_scaled)
Y_test = henon_scaler.inverse_transform(Y_test_scaled)

In [138]:
res_2["data"]["rmes"] = root_mean_squared_error(Y_test, res_2["data"]["Y_pred"])
res_2["data"]["r_2"] = r2_score(Y_test, res_2["data"]["Y_pred"])
res_2["data"]["rmes"], res_2["data"]["r_2"]

(0.05091907143305436, 0.9904280363184079)

In [139]:
henon_plot([Y_test, res_2["data"]["Y_pred"]], names=["Actual", "Predicted"])

### Res 3

In [144]:
X_data = res_3["data"]["state"][transient_steps_reservoir:-tau_steps]
Y_data = henon_scaled[transient_steps_reservoir + tau_steps :]

x_scaler = StandardScaler()
X_train_scaled, X_test_scaled = (
    x_scaler.fit_transform(X_data[:-test_steps]),
    x_scaler.transform(X_data[-test_steps:]),
)
Y_train_scaled, Y_test_scaled = (
    Y_data[:-test_steps],
    Y_data[-test_steps:],
)

In [145]:
try:
    model = Ridge(alpha=res_3["params"]["ridge_alpha"], solver="auto")
    model.fit(X_train_scaled, Y_train_scaled)
except Exception as e:
    try:
        model = Ridge(alpha=res_3["params"]["ridge_alpha"], solver="svd")
        model.fit(X_train_scaled, Y_train_scaled)
        print("Used SVD solver")
    except Exception as e:
        print(f"Error fitting model: {e}")

In [146]:
Y_pred_scaled = model.predict(X_test_scaled)

res_3["data"]["Y_pred"] = henon_scaler.inverse_transform(Y_pred_scaled)
Y_test = henon_scaler.inverse_transform(Y_test_scaled)

In [147]:
res_3["data"]["rmes"] = root_mean_squared_error(Y_test, res_3["data"]["Y_pred"])
res_3["data"]["r_2"] = r2_score(Y_test, res_3["data"]["Y_pred"])
res_3["data"]["rmes"], res_3["data"]["r_2"]

(0.38897002903312783, 0.34484462972203256)

In [148]:
henon_plot([Y_test, res_3["data"]["Y_pred"]], names=["Actual", "Predicted"])

# Bayesian

## Res 1

In [211]:
Y_data = henon_scaled[transient_steps_reservoir + tau_steps :]
Y_train_scaled, Y_test_scaled = (
    Y_data[:-test_steps],
    Y_data[-test_steps:],
)

def henon_closed(
    in_size,
    out_size,
    res_size,
    sparsity,
    spec_rad,
    alpha,
    input_scaling,
    bias_scaling,
    ridge_alpha,
    noise_val,
    tau_steps,
):
    rng = np.random.default_rng(42)
    bias = rng.uniform(
        -bias_scaling,
        bias_scaling,
        res_size,
    )
    W_in = rng.uniform(
        -input_scaling,
        input_scaling,
        (res_size, in_size),
    )
    W_res = rng.uniform(
        -1.0, 1.0, (res_size, res_size)
    )
    mask = (
        rng.random((res_size, res_size))
        < sparsity
    )
    W_res *= mask
    try:
        eigenvalues = la.eigvals(W_res)
    except Exception as e:
        print("Eigenval Issue")
    largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)
    W_res = W_res * (spec_rad / largest_eigenvalue)
    noise = rng.normal(
        0,
        noise_val,
        size=(steps + transient_steps_reservoir, in_size),
    )

    X = compute_states(
        steps + transient_steps_reservoir + tau_steps,
        henon_scaled[:-tau_steps],
        res_size,
        W_in,
        W_res,
        bias,
        alpha,
        noise,
    )

    X_data = X[transient_steps_reservoir:-tau_steps]
    x_scaler = StandardScaler()
    X_train_scaled, X_test_scaled = (
        x_scaler.fit_transform(X_data[:-test_steps]),
        x_scaler.transform(X_data[-test_steps:]),
    )

    with warnings.catch_warnings():
        warnings.filterwarnings("error", category=Warning)
        try:
            model = Ridge(alpha=ridge_alpha, solver="auto")
            model.fit(X_train_scaled, Y_train_scaled)
        except (Warning, ValueError):
            try:
                model = Ridge(alpha=ridge_alpha, solver="svd")
                model.fit(X_train_scaled, Y_train_scaled)
            except Exception as e:
                print("Ridge nor SVD worked")
                raise optuna.exceptions.TrialPruned()

    Y_pred_scaled = model.predict(X_test_scaled)
    Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
    Y_test = henon_scaler.inverse_transform(Y_test_scaled)

    return Y_test, Y_pred

In [221]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=trial.suggest_int("res_size", 50, 500),
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        noise_val=0,
        tau_steps=1,
    )
    rmse = root_mean_squared_error(Y_test, Y_pred)
    std = np.std(Y_test)
    nrmse = rmse / std

    return nrmse


study = optuna.create_study(directions=["minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=100, n_jobs=-1)

[Optuna] Processing Trial #99...

In [233]:
data = []
for i, trial in enumerate(study.best_trials):
    print(f"\r Processing Trial {i}/{len(study.best_trials)}...", end="", flush=True)
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=params["res_size"],
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        noise_val=0.01,
        tau_steps=1,
    )
    data.append((Y_test, Y_pred))

 Processing Trial 0/1...

In [234]:
for Y_test, Y_pred in data:
    print(
        f"Trial {i} - RMSE: {root_mean_squared_error(Y_test, Y_pred)}, R²: {r2_score(Y_test, Y_pred)}"
    )
generate_henon_grid(data, cols=1).show()

Trial 0 - RMSE: 0.00936926148196716, R²: 0.9996712726548367


## Res 2

In [235]:
Y_data = henon_scaled[transient_steps_reservoir + tau_steps :]
Y_train_scaled, Y_test_scaled = (
    Y_data[:-test_steps],
    Y_data[-test_steps:],
)

def henon_closed(
    in_size,
    out_size,
    res_size,
    sparsity,
    spec_rad,
    alpha,
    input_scaling,
    bias_scaling,
    ridge_alpha,
    tau_steps,
    res_size_2,
    sparsity_2,
    spec_rad_2,
    alpha_2,
    input_scaling_2,
    bias_scaling_2,
    noise_val,
):
    rng = np.random.default_rng(42)
    bias = rng.uniform(-bias_scaling, bias_scaling, res_size,)
    W_in = rng.uniform(-input_scaling, input_scaling, (res_size, in_size))
    W_res = rng.uniform(-1.0, 1.0, (res_size, res_size))
    mask = rng.random((res_size, res_size)) < sparsity
    W_res *= mask
    try:
        eigenvalues = la.eigvals(W_res)
    except Exception as e:
        print("Eigenval Issue")
    largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)
    W_res = W_res * (spec_rad / largest_eigenvalue)
    noise = rng.normal(0, noise_val, size=(steps + transient_steps_reservoir, in_size))
    X = compute_states(
        steps + transient_steps_reservoir + tau_steps,
        henon_scaled[:-tau_steps],
        res_size,
        W_in,
        W_res,
        bias,
        alpha,
        noise,
    )

    bias_2 = rng.uniform(-bias_scaling_2, bias_scaling_2, res_size_2,)
    W_in_2 = rng.uniform(-input_scaling_2, input_scaling_2, (res_size_2, res_size))
    W_res_2 = rng.uniform(-1.0, 1.0, (res_size_2, res_size_2))
    mask = rng.random((res_size_2, res_size_2)) < sparsity_2
    W_res_2 *= mask
    try:
        eigenvalues = la.eigvals(W_res_2)
    except Exception as e:
        print("Eigenval Issue")
    largest_eigenvalue = np.round(np.max(np.abs(eigenvalues)), decimals=12)
    W_res_2 = W_res_2 * (spec_rad_2 / largest_eigenvalue)
    noise = np.zeros((steps + transient_steps_reservoir, res_size))
    X_2 = compute_states(
        steps + transient_steps_reservoir + tau_steps,
        X[:-tau_steps],
        res_size_2,
        W_in_2,
        W_res_2,
        bias_2,
        alpha_2,
        noise,
    )

    X_data = X_2[transient_steps_reservoir:-tau_steps]
    x_scaler = StandardScaler()
    X_train_scaled, X_test_scaled = (
        x_scaler.fit_transform(X_data[:-test_steps]),
        x_scaler.transform(X_data[-test_steps:]),
    )

    with warnings.catch_warnings():
        warnings.filterwarnings("error", category=Warning)
        try:
            model = Ridge(alpha=ridge_alpha, solver="auto")
            model.fit(X_train_scaled, Y_train_scaled)
        except (Warning, ValueError):
            try:
                model = Ridge(alpha=ridge_alpha, solver="svd")
                model.fit(X_train_scaled, Y_train_scaled)
            except Exception as e:
                print("Ridge nor SVD worked")
                raise optuna.exceptions.TrialPruned()

    Y_pred_scaled = model.predict(X_test_scaled)
    Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
    Y_test = henon_scaler.inverse_transform(Y_test_scaled)

    return Y_test, Y_pred

In [204]:
Y_test, Y_pred = henon_closed(
    in_size=2,
    out_size=2,
    res_size=res_1["params"]["res_size"],
    sparsity=res_1["params"]["sparsity"],
    spec_rad=res_1["params"]["spec_rad"],
    alpha=res_1["params"]["alpha"],
    input_scaling=res_1["params"]["input_scaling"],
    bias_scaling=res_1["params"]["bias_scaling"],
    ridge_alpha=res_2["params"]["ridge_alpha"],
    tau_steps=1,
    res_size_2=res_2["params"]["res_size"],
    sparsity_2=res_2["params"]["sparsity"],
    spec_rad_2=res_2["params"]["spec_rad"],
    alpha_2=res_2["params"]["alpha"],
    input_scaling_2=res_2["params"]["input_scaling"],
    bias_scaling_2=res_2["params"]["bias_scaling"],
)

In [236]:
def objective(trial):
    print(f"\r[Optuna] Processing Trial #{trial.number}...", end="", flush=True)

    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=trial.suggest_int("res_size", 50, 500),
        sparsity=trial.suggest_float("sparsity", 0.01, 0.5),
        spec_rad=trial.suggest_float("spec_rad", 0.2, 2.0),
        alpha=trial.suggest_float("alpha", 0.1, 0.8),
        input_scaling=trial.suggest_float("input_scaling", 1e-2, 2.0),
        bias_scaling=trial.suggest_float("bias_scaling", 1e-2, 2.0),
        ridge_alpha=trial.suggest_float("ridge_alpha", 1e-13, 1e-3, log=True),
        tau_steps=1,
        res_size_2=trial.suggest_int("res_size_2", 50, 500),
        sparsity_2=trial.suggest_float("sparsity_2", 0.01, 0.5),
        spec_rad_2=trial.suggest_float("spec_rad_2", 0.2, 2.0),
        alpha_2=trial.suggest_float("alpha_2", 0.1, 0.8),
        input_scaling_2=trial.suggest_float("input_scaling_2", 1e-2, 2.0),
        bias_scaling_2=trial.suggest_float("bias_scaling_2", 1e-2, 2.0),
        noise_val=0.01,
    )
    rmse = root_mean_squared_error(Y_test, Y_pred)
    std = np.std(Y_test)
    nrmse = rmse / std

    return nrmse


study = optuna.create_study(directions=["minimize"])
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=300, n_jobs=-1)

[Optuna] Processing Trial #299...

In [242]:
data = []
for i, trial in enumerate(study.best_trials):
    print(f"\r Processing Trial {i}/{len(study.best_trials)}...", end="", flush=True)
    params = trial.params
    Y_test, Y_pred = henon_closed(
        in_size=2,
        out_size=2,
        res_size=params["res_size"],
        sparsity=params["sparsity"],
        spec_rad=params["spec_rad"],
        alpha=params["alpha"],
        input_scaling=params["input_scaling"],
        bias_scaling=params["bias_scaling"],
        ridge_alpha=params["ridge_alpha"],
        tau_steps=1,
        res_size_2=params["res_size_2"],
        sparsity_2=params["sparsity_2"],
        spec_rad_2=params["spec_rad_2"],
        alpha_2=params["alpha_2"],
        input_scaling_2=params["input_scaling_2"],
        bias_scaling_2=params["bias_scaling_2"],
        noise_val=0.01,
    )
    data.append((Y_test, Y_pred))

 Processing Trial 0/1...

In [243]:
for Y_test, Y_pred in data:
    print(
        f"Trial {i} - RMSE: {root_mean_squared_error(Y_test, Y_pred)}, R²: {r2_score(Y_test, Y_pred)}"
    )
generate_henon_grid(data, cols=1).show()

Trial 0 - RMSE: 0.014734543930601574, R²: 0.9991862571620815
